# Chapter 12 — Preserving Provenance · Homework (self-test)

**Goal:** check that you can design claim-source contracts, convert API citations into evidence, classify conflicting values, merge without losing sources, render by content type, and audit a draft before it is published.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 75–105 minutes
- Estimated API cost: **$0** (everything graded runs offline; the exercises use the SDK's real citation types, no requests); the one optional live cell costs < $0.02
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 10 exam-style questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises). This is a practice bar: the real exam reports a scaled score from 100 to 1,000, where 720 passes, which is not the same as 72% of answers correct.

In [ ]:
import hashlib
import json
import os
import re
from typing import Callable, Literal

import anthropic
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field, ValidationError, field_validator

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()  # only the optional live exercise uses it
print("anthropic", anthropic.__version__, "| MODEL =", MODEL)

The optional live cell (Exercise 7) needs a key. Without one, this cell only prints a note: everything graded is offline.

In [ ]:
if not os.getenv("ANTHROPIC_API_KEY"):
    print("No ANTHROPIC_API_KEY: the optional live cell (Exercise 7) will fail; everything graded works offline.")

### Helpers (run, don't edit)

The README's `Source` / `Evidence` / `Claim` models (12.1), plus `publisher` and `source_type` on `Source` and without the optional `confidence` field, a small source registry (`SOURCES`) with methods, dates and one syndication chain (`N1`–`N3` all derive from the press release `PR`), a claim builder `mk(...)`, unit factors and `period_key()` for chronological sorting.

In [ ]:
# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {exc}")


# ---------- the claim contract from the README (12.1) ----------
class Source(BaseModel):
    source_id: str
    title: str
    url: str | None
    publisher: str | None
    source_type: Literal["government", "industry_report", "academic", "news", "company_filing", "internal", "other"]
    published: str | None
    retrieved_at: str | None
    methodology: str | None
    derived_from: str | None


class Evidence(BaseModel):
    source_id: str
    excerpt: str
    location: str | None


class Claim(BaseModel):
    claim_id: str
    statement: str
    subject: str
    metric: str | None
    value: float | None
    unit: str | None
    period: str | None
    evidence: list[Evidence] = Field(min_length=1)


def src(source_id: str, title: str, source_type: str = "industry_report", published: str | None = "2025-01-01",
        methodology: str | None = None, derived_from: str | None = None) -> Source:
    return Source(source_id=source_id, title=title, url=None, publisher=None, source_type=source_type, published=published,
                  retrieved_at=None, methodology=methodology, derived_from=derived_from)


def mk(claim_id: str, value: float | None, unit: str | None, period: str | None, source_id: str, excerpt: str = "...",
       metric: str | None = "market size", subject: str = "AI-generated music market") -> Claim:
    return Claim(claim_id=claim_id, statement=f"{subject} {metric}: {value} {unit} ({period})", subject=subject,
                 metric=metric, value=value, unit=unit, period=period,
                 evidence=[Evidence(source_id=source_id, excerpt=excerpt, location=None)])


SOURCES = {
    "S1": src("S1", "Global AI Music Market Report 2024", published="2024-06-15", methodology="analyst estimate (top-down model)"),
    "S2": src("S2", "Creative Economy Statistics 2025", "government", "2025-02-20", "count of revenue declared in tax filings"),
    "S5": src("S5", "Trade magazine round-up", "news", None, None),
    "X1": src("X1", "Label Panel Survey A", published="2025-03-01", methodology="survey of record labels"),
    "X2": src("X2", "Label Panel Survey B", published="2025-04-01", methodology="survey of record labels"),
    "PR": src("PR", "AI Music Inc. press release", "company_filing", "2025-01-10"),
    "N1": src("N1", "Article 1", "news", "2025-01-11", derived_from="PR"),
    "N2": src("N2", "Article 2", "news", "2025-01-11", derived_from="PR"),
    "N3": src("N3", "Article 3", "news", "2025-01-12", derived_from="PR"),
}
UNIT_FACTORS = {"USD bn": ("USD bn", 1.0), "USD m": ("USD bn", 0.001), "%": ("%", 1.0)}


def period_key(period: str | None) -> tuple:
    """Sortable key for 'YYYY', 'YYYY-MM', 'YYYY-Qn', 'YYYY-Hn'; unknown periods sort last."""
    if not period:
        return (9999, 99)
    m = re.fullmatch(r"(\d{4})(?:-(Q[1-4]|H[12]|\d{2}))?", period)
    if not m:
        return (9999, 98)
    year, part = int(m.group(1)), m.group(2)
    if part is None:
        return (year, 0)
    if part[0] in "QH":
        return (year, int(part[1]) * (3 if part[0] == "Q" else 6))
    return (year, int(part))


print("helpers ready")

## Part A — Quiz (10 questions)

Pick one letter per question. Domains: multi-agent provenance (Domain 5, task 5.6, with 5.1 and 5.3), the Citations API, search results and web citations.

**Q1. (Multi-agent research system)** The synthesis agent's input keeps exceeding its context budget: three subagents each return 15–20 pages of retrieved text plus their reasoning. What is the better fix?

- A) Move synthesis to a model with a larger context window and keep the outputs as they are.
- B) Have each subagent return compact structured claim records (claim, verbatim excerpt, source URL or document name, dates) instead of page dumps and reasoning chains.
- C) Truncate each subagent's output to its first five pages.
- D) Keep the outputs as they are and ask the synthesis agent to cite sources wherever possible.

**Q2. (Multi-agent research system)** The document-analysis subagent finds that official statistics report USD 2.1 billion for 2024, while an analyst report estimates USD 3.2 billion for the same year using a different method. Its prompt says "resolve discrepancies", so it reports USD 2.65 billion. What should change?

- A) Tell it to use the official figure, because government statistics are more authoritative.
- B) Tell it to report the most recently published figure.
- C) Tell it to stop the analysis and wait for the coordinator's decision before continuing.
- D) Have it finish the analysis, keep both values with source, date and method, mark the conflict, and leave reconciliation to the coordinator.

**Q3. (Structured data extraction)** A report published in February 2025 says: "In 2023, 10% of releases used AI tools." Your claim schema has `period` (on the claim), `published` and `retrieved_at` (on the source). Where does "2023" belong?

- A) In `published`, because it is the most important date in the sentence.
- B) Nowhere: the extractor should replace it with the publication year so all figures share one date.
- C) In `period` (the data period), while `published` holds 2025-02.
- D) In `published`, leaving `period` null because one date per claim is enough.

**Q4. (Multi-agent research system)** A finding is labeled "well established (4 sources)". On review, all four sources are news articles quoting the same vendor press release. What is the correct handling?

- A) Record that the articles derive from the press release and count independent origins: it is a single-source finding.
- B) Keep "well established", because four reputable outlets published it.
- C) Drop the finding entirely, because press releases are unreliable.
- D) Raise the threshold for "well established" to five sources.

**Q5. (Multi-agent research system)** Two sub-questions produced no findings: the search on regulatory action ran successfully and returned zero results, and the social-media source timed out after retries. How should the report present them?

- A) Leave both out: a report should contain findings, not non-findings.
- B) Report both as "no information found".
- C) In a coverage section by sub-question, report the first as "searched, no sources found" and the second as "source unavailable, findings may be incomplete".
- D) Treat both as failures: retry the regulatory search until it returns something, and list both under limitations.

**Q6. (Structured data extraction, current API)** One request contains five `document` blocks. The team wants citations only for the three contracts, not for the two style guides, so they enable `citations` on the contracts only. The API rejects the request. What is the right fix?

- A) Retry with exponential backoff; the error is transient.
- B) Enable citations on all five document blocks (or send the style guides as ordinary text content instead of document blocks).
- C) Keep citations enabled on the contracts and set `"citations": {"enabled": false}` explicitly on the style guides.
- D) Add `output_config.format` with a schema that lists which documents may be cited.

**Q7. (Structured data extraction, current API)** A citation on a PDF document is `{"type": "page_location", "start_page_number": 4, "end_page_number": 6, ...}`. Which pages does it cover?

- A) Pages 4, 5 and 6.
- B) Pages 5 and 6.
- C) Pages 5, 6 and 7, because page numbers are 0-indexed.
- D) Pages 4 and 5.

**Q8. (RAG, current API)** Your `kb_search` tool returns `[search_result, search_result, {"type": "text", "text": "Results may be up to 24 hours old."}]` as one `tool_result`'s content, and the request fails validation. What is the right fix?

- A) Move the note into a text block inside one search result's `content` (where it also becomes citable), so the `tool_result` holds only `search_result` blocks.
- B) Put the note before the search results instead of after them.
- C) Disable citations on the search results.
- D) Return each search result in its own `tool_result` with the same `tool_use_id`.

**Q9. (Web research, current API)** To save bandwidth, a team strips `encrypted_content` from `web_search_result` blocks before sending the assistant turn back in the next request. Follow-up requests now fail with a 400. What should they do?

- A) Replace `encrypted_content` with the page's plain text.
- B) Drop the conversation history and re-run the web search on every turn.
- C) Send the assistant content blocks back exactly as received, including `encrypted_content` and `encrypted_index`.
- D) Set `"citations": {"enabled": false}` on the web search tool so the encrypted fields are not needed.

**Q10. (Multi-agent research system)** During synthesis, the synthesis agent often needs to confirm a single number or date. Every check currently goes back through the coordinator to the search subagent, adding about 40% latency. Complex follow-up research is rare. What is the best design?

- A) Give the synthesis agent all of the search subagent's tools.
- B) Give the synthesis agent a narrowly scoped `verify_fact` tool for simple lookups, and keep routing complex investigations through the coordinator.
- C) Let the synthesis agent skip verification when it is confident.
- D) Let the synthesis agent call the search subagent directly for every check, bypassing the coordinator.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365', 'q2': 'd14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b', 'q3': '06217269d5c7af55ec1458d68670a9d1686eee2fa64726f5ac13566f0efad593', 'q4': '0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': '621f4cab9e29d186b9dabeec79a433d2126c30a9e3f5712a1411fc67d73f1e40', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457', 'q8': 'f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc', 'q9': 'e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7', 'q10': 'c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — A subagent output schema that cannot invent metadata

Define two Pydantic models for a research subagent's output (for `client.messages.parse(output_format=SubagentOutput)`):

- `CitedClaim` with `claim`, `excerpt`, `document_name` (strings, **never null**), and `source_url`, `publication_date`, `data_period` (**required but nullable**: the key is always present, the value is `null` when the source does not state it);
- a `field_validator` on `publication_date` that accepts only `YYYY`, `YYYY-MM` or `YYYY-MM-DD` (or `None`) and rejects anything else (`"February 2025"`, `"20/02/2025"`);
- `SubagentOutput` with `claims: list[CitedClaim]`.

The checker inspects `anthropic.transform_schema(SubagentOutput)`, which is exactly what `messages.parse()` sends. Theory: [12.1](README.md#121-the-attribution-loss-problem), [12.3](README.md#123-include-dates-for-correct-interpretation)

In [ ]:
class CitedClaim(BaseModel):
    # TODO: claim, excerpt, source_url, document_name, publication_date (+ validator), data_period
    pass


class SubagentOutput(BaseModel):
    # TODO: claims: list[CitedClaim]
    pass

In [ ]:
def _nullable(prop: dict) -> bool:
    t = prop.get("type")
    return ("null" in t if isinstance(t, list) else t == "null") or any(_nullable(x) for x in prop.get("anyOf", []))


def check_ex1() -> None:
    def check():
        schema = anthropic.transform_schema(SubagentOutput)          # what messages.parse() would send
        assert "claims" in schema.get("properties", {}), "SubagentOutput needs a field `claims: list[CitedClaim]`"
        cc = schema.get("$defs", {}).get("CitedClaim")
        assert cc, "SubagentOutput.claims must be a list of CitedClaim"
        props, required = cc.get("properties", {}), set(cc.get("required", []))
        fields = ["claim", "excerpt", "source_url", "document_name", "publication_date", "data_period"]
        missing = [f for f in fields if f not in props]
        assert not missing, f"CitedClaim is missing fields: {missing}"
        optional = [f for f in fields if f not in required]
        assert not optional, (f"{optional} are not required. Use `str | None` WITHOUT a default (required + nullable): "
                              "a missing key must be an error, `null` must be allowed")
        for f in ("source_url", "publication_date", "data_period"):
            assert _nullable(props[f]), f"{f} must be nullable (`str | None`): internal documents have no URL, many sources no date"
        for f in ("claim", "excerpt", "document_name"):
            assert not _nullable(props[f]), f"{f} must NOT be nullable: a claim without an excerpt or source name is not a claim"
        ok = {"claim": "Revenue was USD 2.1 billion in 2024.", "excerpt": "totalled USD 2.1 billion in 2024",
              "source_url": None, "document_name": "Creative Economy Statistics 2025", "publication_date": None,
              "data_period": "2024"}
        for good in (None, "2025", "2025-02", "2025-02-20"):
            SubagentOutput.model_validate({"claims": [{**ok, "publication_date": good}]})
        for bad in ("February 2025", "20/02/2025", "2025-2-20"):
            try:
                SubagentOutput.model_validate({"claims": [{**ok, "publication_date": bad}]})
            except ValidationError:
                continue
            raise AssertionError(f"publication_date={bad!r} was accepted; add a field_validator that only allows "
                                 "YYYY, YYYY-MM or YYYY-MM-DD (or null)")
        without_date = {k: v for k, v in ok.items() if k != "publication_date"}
        try:
            SubagentOutput.model_validate({"claims": [without_date]})
        except ValidationError:
            return
        raise AssertionError("a claim with NO publication_date key was accepted; the key must be required (value may be null)")
    _run_check(1, check)


check_ex1()

### Exercise 2 — Convert all five citation types into evidence

Write `citation_to_evidence(cit, doc_ids)` returning `{"source_id", "excerpt", "location"}` where `excerpt` is always `cit.cited_text` and `doc_ids[i]` is your source ID for the document at `document_index` `i`:

| `cit.type` | `source_id` | `location` |
|---|---|---|
| `char_location` | `doc_ids[cit.document_index]` | `"chars START-END"` (the API's numbers as given) |
| `page_location` | same | `"p. N"` for one page, else `"pp. A-B"` with **inclusive** B (pages are 1-indexed, `end_page_number` is exclusive) |
| `content_block_location` | same | `"block N"` or `"blocks A-B"` inclusive (end index is exclusive) |
| `search_result_location` | `cit.source` | `"block N"` or `"blocks A-B"` inclusive |
| `web_search_result_location` | `cit.url` | `None` |

Then write `response_to_claims(blocks, doc_ids)` returning `(claims, unsupported)`: one claim `{"statement": text.strip(), "evidence": [...]}` per text block **with** citations, and the stripped text of every text block **without** citations that contains a digit (a number nobody sourced). The checker uses the SDK's real citation classes. Theory: [12.7](README.md#127-the-citations-api-document-blocks-expanded-beyond-the-guide), [12.8](README.md#128-search_result-blocks-citations-for-your-own-retrieval-expanded-beyond-the-guide), [12.9](README.md#129-web-search-and-web-fetch-citations-expanded-beyond-the-guide)

In [ ]:
def citation_to_evidence(cit, doc_ids: list[str]) -> dict:
    """Return {"source_id", "excerpt", "location"} for any of the five citation types (rules above)."""
    # TODO
    raise NotImplementedError


def response_to_claims(blocks: list, doc_ids: list[str]) -> tuple[list[dict], list[str]]:
    """Return (claims, unsupported) from a list of content blocks (rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        T = anthropic.types
        ids = ["S1", "S2"]
        char = T.CitationCharLocation(type="char_location", cited_text="totalled USD 2.1 billion", document_index=1,
                                      document_title="Stats", start_char_index=10, end_char_index=50)
        cases = [
            (char, {"source_id": "S2", "excerpt": "totalled USD 2.1 billion", "location": "chars 10-50"},
             "char_location: map document_index to doc_ids and keep the API's indices as 'chars START-END'"),
            (T.CitationPageLocation(type="page_location", cited_text="p", document_index=0, document_title="R",
                                    start_page_number=3, end_page_number=4),
             {"source_id": "S1", "excerpt": "p", "location": "p. 3"}, "page_location 3..4 (end exclusive) is ONE page: 'p. 3'"),
            (T.CitationPageLocation(type="page_location", cited_text="p", document_index=0, document_title="R",
                                    start_page_number=3, end_page_number=6),
             {"source_id": "S1", "excerpt": "p", "location": "pp. 3-5"}, "page_location 3..6 covers pages 3, 4, 5: 'pp. 3-5'"),
            (T.CitationContentBlockLocation(type="content_block_location", cited_text="b", document_index=1, document_title="T",
                                            start_block_index=0, end_block_index=1),
             {"source_id": "S2", "excerpt": "b", "location": "block 0"}, "content_block_location 0..1 is one block: 'block 0'"),
            (T.CitationContentBlockLocation(type="content_block_location", cited_text="b", document_index=1, document_title="T",
                                            start_block_index=1, end_block_index=3),
             {"source_id": "S2", "excerpt": "b", "location": "blocks 1-2"}, "content_block_location 1..3 is 'blocks 1-2'"),
            (T.CitationsSearchResultLocation(type="search_result_location", cited_text="r", source="kb://policy#v3", title="P",
                                             search_result_index=0, start_block_index=2, end_block_index=3),
             {"source_id": "kb://policy#v3", "excerpt": "r", "location": "block 2"},
             "search_result_location: source_id is cit.source (there is no document_index); location 'block 2'"),
            (T.CitationsWebSearchResultLocation(type="web_search_result_location", cited_text="w", url="https://e.example/a",
                                                title="A", encrypted_index="x"),
             {"source_id": "https://e.example/a", "excerpt": "w", "location": None},
             "web_search_result_location: source_id is the URL, location None"),
        ]
        for cit, expected, hint in cases:
            got = citation_to_evidence(cit, ids)
            assert got == expected, f"{hint}. Got {got}"
        blocks = [T.TextBlock(type="text", text="According to the statistics office, "),
                  T.TextBlock(type="text", text="revenue was USD 2.1 billion in 2024", citations=[char]),
                  T.TextBlock(type="text", text=" (a 10% rise on 2023)"),
                  T.TextBlock(type="text", text=".")]
        result = response_to_claims(blocks, ids)
        assert isinstance(result, tuple) and len(result) == 2, "response_to_claims must return (claims, unsupported)"
        claims, unsupported = result
        assert len(claims) == 1, f"exactly one block has citations, so one claim; got {len(claims)}"
        assert claims[0]["statement"] == "revenue was USD 2.1 billion in 2024", "statement = the cited block's text, stripped"
        assert claims[0]["evidence"] == [cases[0][1]], "evidence = one dict per citation on the block"
        assert unsupported == ["(a 10% rise on 2023)"], (f"uncited text that contains a number must be flagged "
                                                         f"(stripped); text without digits is not. Got {unsupported}")
    _run_check(2, check)


check_ex2()

### Exercise 3 — Tell a trend from a conflict

Write `classify_pair(a, b, sources)` for two `Claim`s. Normalize units first with `UNIT_FACTORS` (`value * factor`, compare base units), then apply these rules **in order**:

1. different subject, metric (case-insensitive) or base unit → `"different_quantity"`
2. either value is `None` → `"not_numeric"`
3. either period is `None` → `"undetermined"`
4. periods differ → `"temporal_difference"`
5. values within 5% (`abs(va - vb) <= 0.05 * max(abs(va), abs(vb))`) → `"agree"`
6. the methodologies of all evidence sources are known and not all equal → `"methodology_difference"`
7. otherwise (including an unknown method) → `"conflict"`

Theory: [12.2](README.md#122-handling-conflicting-data), [12.3](README.md#123-include-dates-for-correct-interpretation)

In [ ]:
def classify_pair(a: Claim, b: Claim, sources: dict[str, Source]) -> str:
    """Return one of: different_quantity, not_numeric, undetermined, temporal_difference, agree, methodology_difference, conflict."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        cases = [
            (mk("A", 3.2, "USD bn", "2024", "S1"), mk("B", 3200, "USD m", "2024", "S2"), "agree",
             "USD 3,200 m and USD 3.2 bn are the same number: normalize units with UNIT_FACTORS first"),
            (mk("A", 3.2, "USD bn", "2024", "S1"), mk("B", 3.3, "USD bn", "2024", "S2"), "agree", "3.2 vs 3.3 is within 5%"),
            (mk("A", 3.2, "USD bn", "2024", "S1"), mk("B", 2.1, "USD bn", "2024", "S2"), "methodology_difference",
             "same period, both methods known and different"),
            (mk("A", 1.9, "USD bn", "2023", "S1"), mk("B", 3.2, "USD bn", "2024", "S1"), "temporal_difference",
             "different periods are a trend, not a contradiction"),
            (mk("A", 1.9, "USD bn", "2023", "S1"), mk("B", 1.9, "USD bn", "2024", "S2"), "temporal_difference",
             "equal values in DIFFERENT periods are a (flat) time series, not 'agree'"),
            (mk("A", 3.2, "USD bn", "2024", "S1"), mk("B", 2.6, "USD bn", None, "S2"), "undetermined",
             "a missing period means you cannot tell a trend from a conflict"),
            (mk("A", 2.4, "USD bn", "2024", "X1"), mk("B", 1.6, "USD bn", "2024", "X2"), "conflict",
             "same period and the SAME method: a genuine conflict"),
            (mk("A", 2.4, "USD bn", "2024", "X1"), mk("B", 1.6, "USD bn", "2024", "S5"), "conflict",
             "an UNKNOWN method is not an explanation: treat it as a conflict"),
            (mk("A", 94, "%", None, "S1", metric="precision"), mk("B", 81, "%", None, "S1", metric="recall"), "different_quantity",
             "precision and recall are different metrics"),
            (mk("A", 12, "%", "2024", "S1"), mk("B", 3.2, "USD bn", "2024", "S1"), "different_quantity",
             "a percentage and a dollar amount are different quantities"),
        ]
        for a, b, expected, hint in cases:
            got = classify_pair(a, b, SOURCES)
            assert got == expected, f"{a.value} {a.unit} ({a.period}, {a.evidence[0].source_id}) vs {b.value} {b.unit} " \
                                    f"({b.period}, {b.evidence[0].source_id}): expected {expected!r}, got {got!r}. Hint: {hint}"
    _run_check(3, check)


check_ex3()

### Exercise 4 — Merge without losing sources, count origins not URLs

Write `merge_claims(claims)`: claims with the same `(subject.lower(), metric.lower(), unit, period, value)` become one claim that keeps the **first** claim's id and statement, with the **union** of evidence (drop exact duplicates of `(source_id, excerpt)`). Keep the order of first appearance and **do not mutate** the input claims.

Write `support_level(claim, sources, contested_ids)`: `"contested"` if the claim id is in `contested_ids`; otherwise count independent origins (`sources[sid].derived_from or sid`): 2 or more → `"well_established"`, else `"single_source"`.

Theory: [12.5](README.md#125-carrying-mappings-through-every-hop-expanded-beyond-the-guide), [12.6](README.md#126-coverage-and-uncertainty-reporting-expanded-beyond-the-guide)

In [ ]:
def merge_claims(claims: list[Claim]) -> list[Claim]:
    """Merge claims with the same subject, metric, unit, period and value; union their evidence (rules above)."""
    # TODO
    raise NotImplementedError


def support_level(claim: Claim, sources: dict[str, Source], contested_ids: set[str]) -> str:
    """Return 'contested', 'well_established' or 'single_source'."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        a = mk("C1", 3.2, "USD bn", "2024", "S1", "We estimate ... USD 3.2 billion in 2024")
        b = mk("C7", 3.2, "USD bn", "2024", "S2", "... totalled USD 3.2 billion in 2024")
        dup = mk("C9", 3.2, "USD bn", "2024", "S1", "We estimate ... USD 3.2 billion in 2024")
        other = mk("C3", 2.1, "USD bn", "2024", "S2", "... USD 2.1 billion in 2024")
        merged = merge_claims([a, b, dup, other])
        assert isinstance(merged, list) and len(merged) == 2, f"expected 2 claims after merging (3.2 and 2.1), got {len(merged)}"
        first = merged[0]
        assert first.claim_id == "C1", "keep the FIRST claim's id (and order of first appearance)"
        assert [(e.source_id, e.excerpt) for e in first.evidence] == [(a.evidence[0].source_id, a.evidence[0].excerpt),
                                                                     (b.evidence[0].source_id, b.evidence[0].excerpt)], \
            "union the evidence lists; drop exact duplicates (same source_id AND excerpt); a merged claim never loses a source"
        assert merged[1].value == 2.1 and len(merged[1].evidence) == 1, "different values stay separate (12.2 classifies them)"
        assert len(a.evidence) == 1, "merge_claims mutated its input; copy claims (model_copy(deep=True)) before appending"

        assert support_level(first, SOURCES, set()) == "well_established", "S1 and S2 are two independent origins"
        assert support_level(other, SOURCES, set()) == "single_source", "one source is single_source"
        syndicated = mk("C5", 2, "million", "2025-01", "N1", "AI Music Inc. says it has 2 million users.", metric="users")
        syndicated.evidence += [Evidence(source_id="N2", excerpt="... 2 million users.", location=None),
                                Evidence(source_id="N3", excerpt="... 2 million users.", location=None)]
        assert support_level(syndicated, SOURCES, set()) == "single_source", \
            "three articles derived_from the same press release are ONE origin: count derived_from or source_id, not URLs"
        assert support_level(first, SOURCES, {"C1"}) == "contested", "a claim in a conflict is 'contested' even if well sourced"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Render financial figures as a sourced table

Write `render_key_figures(claims, sources)` returning a Markdown table (one string):

```text
| Metric | Value | Period | Source (published) | Method |
|---|---|---|---|---|
| <metric> | <value:g> <unit> | <period or "unknown"> | <title> (<published or "date unknown">); ... | <methodology or "not stated">; ... |
```

- only claims with a numeric `value` (others belong in prose);
- rows sorted by metric, then **chronologically** by period (use `period_key`);
- the Source and Method cells list each distinct source of the claim's evidence, in evidence order, joined with `"; "` (methods de-duplicated).

Theory: [12.4](README.md#124-render-by-content-type)

In [ ]:
def render_key_figures(claims: list[Claim], sources: dict[str, Source]) -> str:
    """Return the Markdown key-figures table (exact format above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        claims = [
            mk("C2", 2.1, "USD bn", "2024", "S2"),
            mk("C1", 1.9, "USD bn", "2023", "S1"),
            mk("C4", 7.5, "USD bn", "2027", "S1"),
            mk("C6", None, None, "2024-09", "S5", metric="licensing deals"),
            mk("C9", 12, "%", "2024-H1", "S5", metric="AI share of uploads"),
        ]
        claims[1].evidence.append(Evidence(source_id="S2", excerpt="...", location=None))
        table = render_key_figures(claims, SOURCES)
        assert isinstance(table, str), "return the table as one string"
        lines = [l.strip() for l in table.strip().splitlines()]
        assert lines[0] == "| Metric | Value | Period | Source (published) | Method |", f"header must be exactly as specified, got {lines[0]!r}"
        assert set(lines[1].replace("|", "").strip()) <= {"-", " "}, "second line must be the separator row |---|---|...|"
        rows = [[c.strip() for c in l.strip("|").split("|")] for l in lines[2:]]
        assert len(rows) == 4, f"4 numeric claims -> 4 rows (claims with value None belong in prose, not the table); got {len(rows)}"
        assert [r[0] for r in rows] == ["AI share of uploads", "market size", "market size", "market size"], \
            "sort rows by metric, then chronologically by period (use period_key)"
        assert [r[2] for r in rows[1:]] == ["2023", "2024", "2027"], "market size rows must be in chronological order"
        assert rows[1][1] == "1.9 USD bn", f"Value cell must be '{{value:g}} {{unit}}', got {rows[1][1]!r}"
        assert rows[1][3] == "Global AI Music Market Report 2024 (2024-06-15); Creative Economy Statistics 2025 (2025-02-20)", \
            f"a merged claim lists every source with its publication date, '; '-separated; got {rows[1][3]!r}"
        assert "date unknown" in rows[0][3], "a source without a publication date must say 'date unknown' (never guess)"
        assert rows[0][4] == "not stated", "a missing methodology must say 'not stated'"
        assert rows[1][4] == "analyst estimate (top-down model); count of revenue declared in tax filings", \
            "keep each source's method next to the number (distinct, '; '-separated)"
    _run_check(5, check)


check_ex5()

### Exercise 6 — Audit the draft and annotate coverage

Write `audit_report(sentences, claims, conflicts)` returning a list of problem strings (`[]` = publishable). Claim references look like `[C7]`. For each sentence:

- no reference → one problem containing **"no claim reference"** (skip the other checks);
- a reference not in `claims` → one problem containing **"unknown"** (skip the other checks);
- every number in the sentence (ignore the `[C#]` markers; treat `1,900` as `1900`) must appear in the excerpts or periods of the referenced claims → otherwise a problem containing **"number"**;
- a sentence that references some but not all claim IDs of a conflict (`{"conflict_id", "claim_ids"}`) → a problem containing **"conflict"** (it states one side only).

Write `coverage_lines(items)` (each item: `sub_question`, `topic`, `state`, `detail`) returning one line per **non-covered** item, starting with the sub-question ID: `failed` → mention "unavailable", the detail and that findings "may be incomplete"; `empty` → "no sources found"; `not_searched` → "not examined".

Theory: [12.6](README.md#126-coverage-and-uncertainty-reporting-expanded-beyond-the-guide), [12.10](README.md#1210-verifying-provenance-before-you-publish-expanded-beyond-the-guide)

In [ ]:
def audit_report(sentences: list[str], claims: dict[str, Claim], conflicts: list[dict]) -> list[str]:
    """Return a list of problems; [] means publishable (rules above)."""
    # TODO
    raise NotImplementedError


def coverage_lines(items: list[dict]) -> list[str]:
    """Return one report line per non-covered sub-question (rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    def check():
        claims = {
            "C1": mk("C1", 3.2, "USD bn", "2024", "S1", "We estimate the market at USD 3.2 billion in 2024."),
            "C2": mk("C2", 2.1, "USD bn", "2024", "S2", "Revenue totalled USD 2.1 billion in 2024."),
            "C3": mk("C3", 1.9, "USD bn", "2023", "S2", "Revenue was USD 1900 million in 2023."),
        }
        conflicts = [{"conflict_id": "K1", "claim_ids": ["C1", "C2"]}]
        good = ["Estimates for 2024 differ: USD 3.2 billion [C1] versus USD 2.1 billion [C2].",
                "Registered firms reported USD 1,900 million in 2023 [C3]."]
        got = audit_report(good, claims, conflicts)
        assert got == [], f"well-sourced sentences must pass (commas in numbers are ignored); got {got}"
        bad = [("The market will double by 2026.", "no claim reference"),
               ("Revenue grew sharply [C9].", "unknown"),
               ("Registered firms reported USD 2,000 million in 2023 [C3].", "number"),
               ("Registered firms reported USD 190 million in 2023 [C3].", "number"),  # 190 is inside 1900: match whole numbers
               ("The market was USD 3.2 billion in 2024 [C1].", "conflict")]
        for sentence, keyword in bad:
            got = audit_report([sentence], claims, conflicts)
            assert len(got) == 1 and keyword in got[0].lower(), \
                f"{sentence!r} should give exactly 1 problem mentioning {keyword!r}; got {got}"
        items = [{"sub_question": "Q1", "topic": "market size", "state": "covered", "detail": None},
                 {"sub_question": "Q5", "topic": "listener reception", "state": "failed", "detail": "timeout after 2 attempts"},
                 {"sub_question": "Q6", "topic": "regulation", "state": "empty", "detail": "1 search, 0 results"},
                 {"sub_question": "Q7", "topic": "non-English sources", "state": "not_searched", "detail": None}]
        lines = coverage_lines(items)
        assert isinstance(lines, list) and len(lines) == 3, f"one line per NON-covered sub-question (3); got {lines}"
        failed, empty, skipped = lines
        assert failed.startswith("Q5") and "unavailable" in failed.lower() and "incomplete" in failed.lower() \
            and "timeout" in failed, "a failed source is a GAP: say it was unavailable, why, and that findings may be incomplete"
        assert empty.startswith("Q6") and "no sources found" in empty.lower(), "an empty search is an ANSWER: 'no sources found'"
        assert "unavailable" not in empty.lower(), "do not describe an empty result as unavailable: they mean different things"
        assert skipped.startswith("Q7") and "not examined" in skipped.lower(), "not_searched -> 'not examined'"
    _run_check(6, check)


check_ex6()

### Exercise 7 — your converter on a real cited answer (live, optional, ~$0.01, not scored)

Run it once Exercise 2 passes: two small documents with citations enabled, then your `response_to_claims` and a slice-based integrity check on the `char_location` citations.

In [ ]:
# (live, optional, ~$0.01, not scored)
DOCS = [
    ("S1", "Global AI Music Market Report 2024",
     "We estimate the global market for AI-generated music at USD 3.2 billion in 2024. We project USD 7.5 billion by 2027."),
    ("S2", "Creative Economy Statistics 2025",
     "Revenue reported by registered firms totalled USD 2.1 billion in 2024. Figures for 2024 are preliminary."),
]
if not RESULTS.get(2):
    print("Finish Exercise 2 first (its checker must pass), then run this cell.")
else:
    content = [{"type": "document", "source": {"type": "text", "media_type": "text/plain", "data": text}, "title": title,
                "context": json.dumps({"source_id": sid}), "citations": {"enabled": True}} for sid, title, text in DOCS]
    content.append({"type": "text", "text": "How large was the AI-generated music market in 2024? Give each source's figure."})
    r = client.messages.create(model=MODEL, max_tokens=600, messages=[{"role": "user", "content": content}])
    claims, unsupported = response_to_claims(r.content, [d[0] for d in DOCS])
    for c in claims:
        print(c["statement"], "->", c["evidence"])
    print("unsupported:", unsupported)
    bad = [c for b in r.content if b.type == "text" for c in (b.citations or []) if c.type == "char_location"
           and DOCS[c.document_index][2][c.start_char_index:c.end_char_index].strip() != c.cited_text.strip()]
    print("integrity:", "OK" if not bad else bad)

## Part C — Architecture scenario

**Investment due-diligence briefs.** An investment firm wants a multi-agent system (on `claude-sonnet-5-5`) that produces a 3–5 page brief on a target company for its investment committee within an hour. Inputs: the company's annual reports and filings (PDFs, hundreds of pages), recent web news, analyst notes in an internal knowledge base, and two paid data feeds that are often slow. Committee members have complained that earlier AI briefs contained revenue figures nobody could trace, mixed fiscal years, and once presented a rumor as fact. Compliance requires that every figure in a brief can be traced to its source for seven years.

Describe your design (15–25 lines):
1. The agents and what flows between them (the data contract, how it is passed, who reconciles what).
2. Which platform features produce provenance on each hop (documents, knowledge base, web), and where you must rely on schemas instead.
3. How the brief handles conflicting figures, fiscal-year and date issues, and sources that failed or returned nothing.
4. The checks that run before a brief reaches the committee, and where humans are involved.
5. What you store for the seven-year traceability requirement.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Hub-and-spoke**: a coordinator with scoped subagents (filings analysis, KB analysis, web news, data feeds) running in parallel; the coordinator controls flow, tracks coverage and reconciles conflicts; no direct subagent-to-synthesis bypass.
- **Claim records as the contract** on every hop: statement (with the source's qualifiers), verbatim excerpt, source ID/URL/document name, location (page), **data period / fiscal year**, publication date, retrieval date, methodology; required-but-nullable metadata, `null` instead of guesses; subagents return structured records, not prose; large batches passed **by reference** (claim store), not retold; no intermediate summarizer.
- **Platform provenance**: filings as PDF `document` blocks with citations (`page_location`, 1-indexed, end exclusive), metadata in `context`, prompt caching on the large PDFs; the KB as `search_result` blocks with paragraph-sized content blocks and versioned `source` IDs; web search (citations always on, `page_age`) plus web fetch with citations for key articles, `allowed_domains` for trusted outlets. Citations and structured outputs are not combined in one request: cite first, convert in code, then a separate structured call for fields. Subagent → coordinator hops rely on the schema.
- **Conflicts and dates**: normalize units and fiscal-year definitions, classify (temporal, definition/scope, methodology, genuine); keep both values with attribution in a contested section; coordinator reconciles or requests verification; fiscal year vs calendar year and publication vs data period kept apart; resolve relative dates against publication dates; rumors stay labeled as such (qualifiers preserved) and single-source.
- **Coverage**: per sub-question `covered` / `empty` / `failed` / `not_searched` (slow feeds time out → structured error context, partial results kept, "findings may be incomplete" in the brief), single-source findings listed; support levels from independent origins, not self-rated confidence.
- **Rendering**: financial figures in code-rendered tables (value, unit, fiscal period, source with date, method), news as prose with inline citations, risks/technical points as lists, a timeline for multi-year series, numbered references.
- **Checks**: code audit (every sentence has claim IDs, numbers appear in evidence, contested figures never stated one-sided, gaps mentioned), integrity checks on citation offsets, a fresh-instance reviewer that retracts unsupported claims, a scoped `verify_fact` tool for quick checks, and **human review** (analyst sign-off) for financial figures that drive the decision and for contested or single-source material findings.
- **Traceability storage**: claim store, source registry with retrieval timestamps and document versions/hashes, the cited responses and the rendered brief, audit results and reviewer decisions, retained seven years; untrusted web content treated as data (prompt-injection defense), never as instructions.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 6
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to Chapter 13" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next chapter: [Chapter 13 — Claude Code Built-in Tools](../13-claude-code-builtin-tools/README.md). Back to the [Claude path overview](../README.md).

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **B.** Structured claim records keep the evidence and cut the volume (task 5.1). A bigger window (A) keeps the page dumps, so attribution stays buried in noise and the budget problem returns. C loses findings arbitrarily. D asks the writer for links it no longer has and leaves the context problem unsolved.

**Q2:** **D.** Complete the analysis, keep both values with full attribution, annotate the conflict, and let the coordinator reconcile (task 5.6). A and B silently pick a winner. C blocks the pipeline for a decision that can wait. Averaging produced a number that no source reported.

**Q3:** **C.** "2023" is the data period: what time the number describes. `published` (2025-02) says how recent the source is, and `retrieved_at` says when your system read it. B fabricates a date, and A/D conflate two different dates, which is what makes trends look like contradictions.

**Q4:** **A.** Count independent origins, not URLs: syndicated or derivative articles are one source. Record lineage (`derived_from`) so the count is correct. B is the error itself. C throws away a real (single-source) finding; label it instead. D does not fix the counting.

**Q5:** **C.** `empty` (searched, no matches) is an answer; `failed` (could not look) is a gap. Both belong in a coverage section by sub-question. A misleads by omission, B merges two different situations, and D treats a valid empty result as a failure and wastes searches on it.

**Q6:** **B.** Citations must be enabled on all document blocks in a request or on none. C is still a mixed setting, so it fails the same way. A cannot fix a deterministic validation error. D makes it worse: citations plus `output_config.format` is itself a 400.

**Q7:** **D.** Page numbers are 1-indexed and `end_page_number` is exclusive, so 4..6 covers pages 4 and 5. (Character and block indices are 0-indexed, also with exclusive ends.)

**Q8:** **A.** If any block in a `tool_result` is a `search_result`, all must be. Supporting text goes inside a search result's `content`, where it is also citable. B keeps the mixing. C leaves the plain text block in the `tool_result`, so the mixing error remains; citation settings are unrelated. D breaks the one-result-per-`tool_use_id` protocol.

**Q9:** **C.** Send assistant content back unchanged; the API decrypts `encrypted_content` to restore the search results in Claude's context, and a missing or modified value returns a 400. A modifies the block. B wastes searches and loses context. D is impossible: web search citations are always enabled.

**Q10:** **B.** A narrowly scoped `verify_fact` tool handles the frequent simple case cheaply, and the coordinator still handles complex investigations (keeping visibility and control). A over-provisions tools. C skips verification based on self-confidence. D gives up the coordinator's control entirely.
</details>

<details><summary>Exercise 1 — A subagent output schema that cannot invent metadata: reference solution</summary>

```python
class CitedClaim(BaseModel):
    claim: str
    excerpt: str
    source_url: str | None            # required key, null for internal documents
    document_name: str
    publication_date: str | None      # required key, null when the source does not state it
    data_period: str | None

    @field_validator("publication_date")
    @classmethod
    def iso_or_null(cls, v: str | None) -> str | None:
        if v is not None and not re.fullmatch(r"\d{4}(-\d{2}(-\d{2})?)?", v):
            raise ValueError("publication_date must be YYYY, YYYY-MM or YYYY-MM-DD, or null")
        return v


class SubagentOutput(BaseModel):
    claims: list[CitedClaim]
```
</details>

<details><summary>Exercise 2 — Convert all five citation types into evidence: reference solution</summary>

```python
def _span(start: int, end_exclusive: int, one: str, many: str) -> str:
    last = end_exclusive - 1
    return f"{one} {start}" if last == start else f"{many} {start}-{last}"


def citation_to_evidence(cit, doc_ids: list[str]) -> dict:
    match cit.type:
        case "char_location":
            return {"source_id": doc_ids[cit.document_index], "excerpt": cit.cited_text,
                    "location": f"chars {cit.start_char_index}-{cit.end_char_index}"}
        case "page_location":
            return {"source_id": doc_ids[cit.document_index], "excerpt": cit.cited_text,
                    "location": _span(cit.start_page_number, cit.end_page_number, "p.", "pp.")}
        case "content_block_location":
            return {"source_id": doc_ids[cit.document_index], "excerpt": cit.cited_text,
                    "location": _span(cit.start_block_index, cit.end_block_index, "block", "blocks")}
        case "search_result_location":
            return {"source_id": cit.source, "excerpt": cit.cited_text,
                    "location": _span(cit.start_block_index, cit.end_block_index, "block", "blocks")}
        case "web_search_result_location":
            return {"source_id": cit.url, "excerpt": cit.cited_text, "location": None}
    raise ValueError(f"unknown citation type {cit.type}")


def response_to_claims(blocks: list, doc_ids: list[str]) -> tuple[list[dict], list[str]]:
    claims, unsupported = [], []
    for b in blocks:
        if b.type != "text":
            continue
        if b.citations:
            claims.append({"statement": b.text.strip(), "evidence": [citation_to_evidence(c, doc_ids) for c in b.citations]})
        elif re.search(r"\d", b.text):
            unsupported.append(b.text.strip())
    return claims, unsupported
```
</details>

<details><summary>Exercise 3 — Tell a trend from a conflict: reference solution</summary>

```python
def _norm(c: Claim) -> tuple[float | None, str | None]:
    if c.value is None or c.unit is None:
        return c.value, c.unit
    base, factor = UNIT_FACTORS.get(c.unit, (c.unit, 1.0))
    return c.value * factor, base


def classify_pair(a: Claim, b: Claim, sources: dict[str, Source]) -> str:
    (va, ua), (vb, ub) = _norm(a), _norm(b)
    if (a.subject.lower(), (a.metric or "").lower(), ua) != (b.subject.lower(), (b.metric or "").lower(), ub):
        return "different_quantity"
    if va is None or vb is None:
        return "not_numeric"
    if a.period is None or b.period is None:
        return "undetermined"
    if a.period != b.period:
        return "temporal_difference"
    if abs(va - vb) <= 0.05 * max(abs(va), abs(vb)):
        return "agree"
    methods = {sources[e.source_id].methodology for e in a.evidence + b.evidence}
    if None not in methods and len(methods) > 1:
        return "methodology_difference"
    return "conflict"
```
</details>

<details><summary>Exercise 4 — Merge without losing sources, count origins not URLs: reference solution</summary>

```python
def merge_claims(claims: list[Claim]) -> list[Claim]:
    merged: dict[tuple, Claim] = {}
    for c in claims:
        k = (c.subject.lower(), (c.metric or "").lower(), c.unit, c.period, c.value)
        if k not in merged:
            merged[k] = c.model_copy(deep=True)          # never mutate the inputs
            continue
        seen = {(e.source_id, e.excerpt) for e in merged[k].evidence}
        merged[k].evidence += [e.model_copy() for e in c.evidence if (e.source_id, e.excerpt) not in seen]
    return list(merged.values())


def support_level(claim: Claim, sources: dict[str, Source], contested_ids: set[str]) -> str:
    if claim.claim_id in contested_ids:
        return "contested"
    origins = {sources[e.source_id].derived_from or e.source_id for e in claim.evidence}
    return "well_established" if len(origins) >= 2 else "single_source"
```
</details>

<details><summary>Exercise 5 — Render financial figures as a sourced table: reference solution</summary>

```python
def render_key_figures(claims: list[Claim], sources: dict[str, Source]) -> str:
    rows = sorted((c for c in claims if c.value is not None), key=lambda c: ((c.metric or ""), period_key(c.period)))
    lines = ["| Metric | Value | Period | Source (published) | Method |", "|---|---|---|---|---|"]
    for c in rows:
        ids = list(dict.fromkeys(e.source_id for e in c.evidence))
        src = "; ".join(f"{sources[s].title} ({sources[s].published or 'date unknown'})" for s in ids)
        method = "; ".join(dict.fromkeys(sources[s].methodology or "not stated" for s in ids))
        lines.append(f"| {c.metric} | {c.value:g} {c.unit} | {c.period or 'unknown'} | {src} | {method} |")
    return "\n".join(lines)
```
</details>

<details><summary>Exercise 6 — Audit the draft and annotate coverage: reference solution</summary>

```python
REF = re.compile(r"\[(C\d+)\]")
NUM = re.compile(r"\d+(?:[.,]\d+)*")


def audit_report(sentences: list[str], claims: dict[str, Claim], conflicts: list[dict]) -> list[str]:
    problems = []
    for s in sentences:
        refs = REF.findall(s)
        if not refs:
            problems.append(f"no claim reference: {s}")
            continue
        unknown = [r for r in refs if r not in claims]
        if unknown:
            problems.append(f"unknown claim IDs {unknown}: {s}")
            continue
        allowed_text = " ".join(e.excerpt for r in refs for e in claims[r].evidence)
        allowed_text += " " + " ".join(claims[r].period or "" for r in refs)
        allowed = {m.replace(",", "") for m in NUM.findall(allowed_text)}  # whole numbers, not substrings
        for n in NUM.findall(REF.sub("", s)):
            if n.replace(",", "") not in allowed:
                problems.append(f"number {n} not found in the evidence of {refs}: {s}")
        for k in conflicts:
            ids = set(k["claim_ids"])
            if ids & set(refs) and not ids <= set(refs):
                problems.append(f"states only one side of conflict {k['conflict_id']}: {s}")
    return problems


def coverage_lines(items: list[dict]) -> list[str]:
    out = []
    for it in items:
        q, topic, detail = it["sub_question"], it["topic"], it.get("detail") or ""
        if it["state"] == "failed":
            out.append(f"{q} ({topic}): sources were unavailable ({detail}); findings may be incomplete.")
        elif it["state"] == "empty":
            out.append(f"{q} ({topic}): searched, no sources found ({detail}).")
        elif it["state"] == "not_searched":
            out.append(f"{q} ({topic}): not examined in this report.")
    return out
```
</details>